# Calculate Design Temperature from Daily Temperature Data

This notebook calculates design temperature for each station from daily temperature data.

**Process:**
1. Read daily temperature files (with lat/long coordinates)
2. Filter data to only include years 2003 onwards (last 22 years)
3. Calculate 2-day rolling average of daily mean temperature (TT_TU_mean)
4. Find the 10 lowest values of these 2-day rolling averages
5. Design temperature is the lowest of these 10 values
6. Save design temperature for each station in a separate folder

In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import traceback

In [2]:
# Configuration
INPUT_BASE_DIR = Path("/mnt/d/heatpump_data/climate_data/dwd_historical_daily_temperature_extracted_csv")
OUTPUT_BASE_DIR = Path("/mnt/d/heatpump_data/climate_data/dwd_design_temperature")
MIN_YEAR = 2003  # Only process data from 2003 onwards (last 22 years)

print(f"Input directory: {INPUT_BASE_DIR}")
print(f"Input directory exists: {INPUT_BASE_DIR.exists()}")
print(f"Output directory: {OUTPUT_BASE_DIR}")
print(f"Output directory exists: {OUTPUT_BASE_DIR.exists()}")
print(f"Minimum year to process: {MIN_YEAR}")

# Create output directory if it doesn't exist
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# Note: Output files will be saved maintaining the same folder structure as input
# Input: /mnt/d/heatpump_data/climate_data/dwd_historical_daily_temperature_extracted_csv/{station_dir}/{file}.csv
# Output: /mnt/d/heatpump_data/climate_data/dwd_design_temperature/{station_dir}/{file}.csv

Input directory: /mnt/d/heatpump_data/climate_data/dwd_historical_daily_temperature_extracted_csv
Input directory exists: True
Output directory: /mnt/d/heatpump_data/climate_data/dwd_design_temperature
Output directory exists: False
Minimum year to process: 2003


In [3]:
def calculate_design_temperature(daily_df):
    """
    Calculate design temperature from daily temperature data.
    
    Design temperature is calculated as:
    1. Filter data to years >= 2003
    2. Calculate 2-day rolling average of TT_TU_mean
    3. Find the 10 lowest values of rolling averages
    4. Design temperature is the lowest of these 10 values
    
    Args:
        daily_df: DataFrame with daily temperature data containing MESS_DATUM, TT_TU_mean columns
        
    Returns:
        Dictionary with design temperature and metadata, or None if insufficient data
    """
    if len(daily_df) == 0:
        return None
    
    # Create a copy to avoid modifying original
    df = daily_df.copy()
    
    # Convert MESS_DATUM to datetime
    df['date'] = pd.to_datetime(df['MESS_DATUM'].astype(str), format='%Y%m%d', errors='coerce')
    
    # Remove rows with invalid dates
    df = df[df['date'].notna()].copy()
    
    if len(df) == 0:
        return None
    
    # Filter to only include years >= 2003
    df['year'] = df['date'].dt.year
    df_filtered = df[df['year'] >= MIN_YEAR].copy()
    
    if len(df_filtered) == 0:
        return None
    
    # Sort by date
    df_filtered = df_filtered.sort_values('date').reset_index(drop=True)
    
    # Get station metadata (should be constant per file)
    station_id = df_filtered['STATIONS_ID'].iloc[0] if 'STATIONS_ID' in df_filtered.columns else None
    latitude = df_filtered['latitude'].iloc[0] if 'latitude' in df_filtered.columns else None
    longitude = df_filtered['longitude'].iloc[0] if 'longitude' in df_filtered.columns else None
    
    # Ensure TT_TU_mean is numeric
    if 'TT_TU_mean' not in df_filtered.columns:
        return None
    
    df_filtered['TT_TU_mean'] = pd.to_numeric(df_filtered['TT_TU_mean'], errors='coerce')
    
    # Remove rows with missing temperature data
    df_filtered = df_filtered[df_filtered['TT_TU_mean'].notna()].copy()
    
    if len(df_filtered) < 2:
        return None  # Need at least 2 days for rolling average
    
    # Calculate 2-day rolling average
    # Average of day i and day i+1
    df_filtered['rolling_avg_2day'] = (
        df_filtered['TT_TU_mean'] + df_filtered['TT_TU_mean'].shift(-1)
    ) / 2
    
    # Remove NaN values (last day won't have a pair)
    df_with_rolling = df_filtered[df_filtered['rolling_avg_2day'].notna()].copy()
    
    if len(df_with_rolling) < 10:
        return None  # Need at least 10 rolling averages
    
    # Find the 10 lowest values of rolling averages
    lowest_10 = df_with_rolling.nsmallest(10, 'rolling_avg_2day')
    
    # Design temperature is the lowest of these 10 values
    design_temperature = lowest_10['rolling_avg_2day'].min()
    
    return {
        'STATIONS_ID': station_id,
        'latitude': latitude,
        'longitude': longitude,
        'design_temperature': design_temperature,
        'date_range_start': df_filtered['date'].min().strftime('%Y%m%d'),
        'date_range_end': df_filtered['date'].max().strftime('%Y%m%d'),
        'total_days': len(df_filtered),
        'days_with_rolling_avg': len(df_with_rolling),
        'lowest_10_rolling_avgs': lowest_10['rolling_avg_2day'].tolist()
    }

In [4]:
def process_station_file(daily_file_path, output_base_dir):
    """
    Process a single daily temperature file and calculate design temperature.
    
    Args:
        daily_file_path: Path to daily temperature CSV file
        output_base_dir: Base directory for output files
        
    Returns:
        Dictionary with processing results
    """
    try:
        # Read daily file
        daily_df = pd.read_csv(daily_file_path)
        
        if len(daily_df) == 0:
            return {
                'status': 'skipped',
                'reason': 'Empty file',
                'file': daily_file_path.name
            }
        
        # Calculate design temperature
        design_result = calculate_design_temperature(daily_df)
        
        if design_result is None:
            return {
                'status': 'skipped',
                'reason': 'Insufficient data or no data after 2003',
                'file': daily_file_path.name
            }
        
        # Get relative path from input base directory
        relative_path = daily_file_path.relative_to(INPUT_BASE_DIR)
        
        # Create output path maintaining same folder structure
        output_file_path = output_base_dir / relative_path
        
        # Create parent directories if they don't exist
        output_file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Create output DataFrame with design temperature
        # Format the 10 minimum values as a comma-separated string
        lowest_10_str = ', '.join([f'{x:.2f}' for x in design_result['lowest_10_rolling_avgs']])
        
        output_df = pd.DataFrame([{
            'STATIONS_ID': design_result['STATIONS_ID'],
            'latitude': design_result['latitude'],
            'longitude': design_result['longitude'],
            'design_temperature': design_result['design_temperature'],
            'lowest_10_rolling_avgs': lowest_10_str,
            'date_range_start': design_result['date_range_start'],
            'date_range_end': design_result['date_range_end'],
            'total_days': design_result['total_days'],
            'days_with_rolling_avg': design_result['days_with_rolling_avg']
        }])
        
        # Save design temperature data
        output_df.to_csv(output_file_path, index=False)
        
        return {
            'status': 'success',
            'file': daily_file_path.name,
            'output_file': output_file_path.name,
            'station_id': design_result['STATIONS_ID'],
            'design_temperature': design_result['design_temperature'],
            'date_range': f"{design_result['date_range_start']} to {design_result['date_range_end']}",
            'total_days': design_result['total_days']
        }
        
    except Exception as e:
        return {
            'status': 'error',
            'error': str(e),
            'file': daily_file_path.name,
            'traceback': traceback.format_exc()
        }

## Test: Process a Single File

Test the design temperature calculation on a single file before processing all files.

In [5]:
# Test: Calculate design temperature for a single file
print("=" * 80)
print("TEST: Calculating design temperature for a single file")
print("=" * 80)

# Find one daily file with lat/long coordinates
test_files = list(INPUT_BASE_DIR.rglob("*_lat_*_lon_*.csv"))
if len(test_files) == 0:
    print("No files found with lat/long coordinates!")
else:
    # Select a file that likely has data from 2003 onwards
    test_file = None
    for f in test_files:
        # Try to find a file with recent dates in the filename
        if '2003' in f.name or '2004' in f.name:
            test_file = f
            break
    
    if test_file is None:
        test_file = test_files[0]
    
    print(f"\nTest file: {test_file.name}")
    print(f"Full path: {test_file}")
    
    # Read and display sample of daily data
    print("\n" + "-" * 80)
    print("DAILY DATA (Input)")
    print("-" * 80)
    daily_df = pd.read_csv(test_file)
    print(f"Total daily records: {len(daily_df):,}")
    print(f"Columns: {', '.join(daily_df.columns)}")
    
    # Convert MESS_DATUM to datetime for display
    daily_df['date'] = pd.to_datetime(daily_df['MESS_DATUM'].astype(str), format='%Y%m%d', errors='coerce')
    valid_dates = daily_df[daily_df['date'].notna()]
    if len(valid_dates) > 0:
        print(f"Date range: {valid_dates['date'].min()} to {valid_dates['date'].max()}")
        print(f"Year range: {valid_dates['date'].dt.year.min()} to {valid_dates['date'].dt.year.max()}")
        records_2003_plus = valid_dates[valid_dates['date'].dt.year >= MIN_YEAR]
        print(f"Records from {MIN_YEAR} onwards: {len(records_2003_plus):,}")
    
    print(f"\nFirst 10 rows:")
    print(daily_df.head(10))
    
    # Calculate design temperature
    print("\n" + "-" * 80)
    print("CALCULATING DESIGN TEMPERATURE...")
    print("-" * 80)
    design_result = calculate_design_temperature(daily_df)
    
    if design_result:
        print(f"\nDesign Temperature Result:")
        print(f"  Station ID: {design_result['STATIONS_ID']}")
        print(f"  Latitude: {design_result['latitude']}")
        print(f"  Longitude: {design_result['longitude']}")
        print(f"  Design Temperature: {design_result['design_temperature']:.2f}°C")
        print(f"  Date range: {design_result['date_range_start']} to {design_result['date_range_end']}")
        print(f"  Total days (>=2003): {design_result['total_days']:,}")
        print(f"  Days with rolling avg: {design_result['days_with_rolling_avg']:,}")
        print(f"  Lowest 10 rolling averages: {[f'{x:.2f}' for x in design_result['lowest_10_rolling_avgs']]}")
    else:
        print("\nCould not calculate design temperature (insufficient data or no data after 2003)")
    
    # Test saving the file
    print("\n" + "-" * 80)
    print("TESTING FILE SAVE...")
    print("-" * 80)
    result = process_station_file(test_file, OUTPUT_BASE_DIR)
    print(f"Status: {result['status']}")
    if result['status'] == 'success':
        relative_path = test_file.relative_to(INPUT_BASE_DIR)
        output_file_path = OUTPUT_BASE_DIR / relative_path
        print(f"Output file saved to: {output_file_path}")
        print(f"Output file exists: {output_file_path.exists()}")
        if output_file_path.exists():
            df_check = pd.read_csv(output_file_path)
            print(f"\nOutput file contents:")
            print(df_check)
    else:
        print(f"Error/Skip reason: {result.get('reason', result.get('error', 'Unknown'))}")
    
    print("\n" + "=" * 80)
    print("TEST COMPLETE")
    print("=" * 80)

TEST: Calculating design temperature for a single file

Test file: produkt_tu_stunde_20041101_20241231_78_4_lat_52_5026_lon_7_9468.csv
Full path: /mnt/d/heatpump_data/climate_data/dwd_historical_daily_temperature_extracted_csv/stundenwerte_TU_00078_20041101_20241231_hist/produkt_tu_stunde_20041101_20241231_78_4_lat_52_5026_lon_7_9468.csv

--------------------------------------------------------------------------------
DAILY DATA (Input)
--------------------------------------------------------------------------------
Total daily records: 7,366
Columns: STATIONS_ID, MESS_DATUM, latitude, longitude, TT_TU_mean, TT_TU_min, TT_TU_max, TT_TU_count, QN_9
Date range: 2004-11-01 00:00:00 to 2024-12-31 00:00:00
Year range: 2004 to 2024
Records from 2003 onwards: 7,366

First 10 rows:
  STATIONS_ID  MESS_DATUM  latitude  longitude  TT_TU_mean  TT_TU_min  \
0        78_4    20041101   52.5026     7.9468    7.108333        3.3   
1        78_4    20041102   52.5026     7.9468    7.579167        5.1

In [6]:
# Find all station directories (folders) containing daily temperature files
print("=" * 80)
print("Finding all station directories with daily temperature files...")
print("=" * 80)

# Get all station directories
station_dirs = [d for d in INPUT_BASE_DIR.iterdir() if d.is_dir()]
station_dirs.sort()

# Filter to only directories that have lat/lon files
station_dirs_with_files = []
for station_dir in station_dirs:
    lat_lon_files = list(station_dir.glob("*_lat_*_lon_*.csv"))
    if len(lat_lon_files) > 0:
        station_dirs_with_files.append((station_dir, lat_lon_files))

print(f"Found {len(station_dirs_with_files)} station directories with daily temperature files")
total_files = sum(len(files) for _, files in station_dirs_with_files)
print(f"Total files to process: {total_files:,}")

# Display summary by station
print(f"\nFiles per station directory:")
for station_dir, files in station_dirs_with_files[:10]:
    print(f"  {station_dir.name}: {len(files)} files")
if len(station_dirs_with_files) > 10:
    print(f"  ... and {len(station_dirs_with_files) - 10} more stations")

Finding all station directories with daily temperature files...
Found 555 station directories with daily temperature files
Total files to process: 1,004

Files per station directory:
  stundenwerte_TU_00003_19500401_20110331_hist: 2 files
  stundenwerte_TU_00044_20070401_20241231_hist: 1 files
  stundenwerte_TU_00052_19760101_19880101_hist: 1 files
  stundenwerte_TU_00071_20091201_20191231_hist: 1 files
  stundenwerte_TU_00073_20070401_20241231_hist: 2 files
  stundenwerte_TU_00078_20041101_20241231_hist: 1 files
  stundenwerte_TU_00091_20040901_20241231_hist: 1 files
  stundenwerte_TU_00096_20190409_20241231_hist: 1 files
  stundenwerte_TU_00102_20020101_20241231_hist: 1 files
  stundenwerte_TU_00125_19710104_20241231_hist: 3 files
  ... and 545 more stations


In [7]:
# Process all daily files in batches by station folder
print("=" * 80)
print("Processing daily files in batches by station folder...")
print("=" * 80)

results = []
success_count = 0
error_count = 0
skipped_count = 0

# Process each station directory (batch)
for station_dir, daily_files in tqdm(station_dirs_with_files, desc="Processing stations", unit="station"):
    station_results = []
    station_success = 0
    station_error = 0
    station_skipped = 0
    
    # Process all files in this station directory
    for daily_file in daily_files:
        result = process_station_file(daily_file, OUTPUT_BASE_DIR)
        result['input_file_path'] = str(daily_file)
        result['station_dir'] = station_dir.name
        results.append(result)
        station_results.append(result)
        
        if result['status'] == 'success':
            success_count += 1
            station_success += 1
        elif result['status'] == 'error':
            error_count += 1
            station_error += 1
        elif result['status'] == 'skipped':
            skipped_count += 1
            station_skipped += 1

print("\n" + "=" * 80)
print("Processing complete!")
print("=" * 80)

Processing daily files in batches by station folder...


Processing stations: 100%|██████████| 555/555 [00:40<00:00, 13.67station/s]


Processing complete!


In [8]:
# Summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print(f"\nTotal files processed: {len(results)}")
print(f"  Successfully processed: {success_count}")
print(f"  Errors: {error_count}")
print(f"  Skipped: {skipped_count}")

# Aggregate statistics from successful processing
successful_results = [r for r in results if r['status'] == 'success']

if successful_results:
    unique_stations = len(set(r.get('station_id') for r in successful_results if r.get('station_id') is not None))
    design_temps = [r.get('design_temperature') for r in successful_results if r.get('design_temperature') is not None]
    
    print(f"\nDesign Temperature Statistics:")
    print(f"  Unique stations: {unique_stations}")
    if design_temps:
        print(f"  Min design temp: {min(design_temps):.2f}°C")
        print(f"  Max design temp: {max(design_temps):.2f}°C")
        print(f"  Mean design temp: {sum(design_temps)/len(design_temps):.2f}°C")

# Show files with errors
error_results = [r for r in results if r['status'] == 'error']
if error_results:
    print(f"\n{'=' * 80}")
    print(f"FILES WITH ERRORS ({len(error_results)}):")
    print("=" * 80)
    for r in error_results[:10]:  # Show first 10 errors
        print(f"\nFile: {r['file']}")
        print(f"Error: {r['error']}")
    if len(error_results) > 10:
        print(f"\n... and {len(error_results) - 10} more errors")


SUMMARY STATISTICS

Total files processed: 1004
  Successfully processed: 746
  Errors: 0
  Skipped: 258

Design Temperature Statistics:
  Unique stations: 746
  Min design temp: -27.13°C
  Max design temp: 0.82°C
  Mean design temp: -12.41°C


In [9]:
# Sample verification - check a few calculated design temperatures
print("\n" + "=" * 80)
print("SAMPLE VERIFICATION")
print("=" * 80)

sample_results = [r for r in successful_results[:5]]

for result in sample_results:
    # Get output file path
    relative_path = Path(result['input_file_path']).relative_to(INPUT_BASE_DIR)
    output_file_path = OUTPUT_BASE_DIR / relative_path
    
    print(f"\nStation: {result.get('station_id', 'N/A')}")
    print(f"  Input file: {result['file']}")
    print(f"  Design Temperature: {result.get('design_temperature', 'N/A'):.2f}°C")
    print(f"  Date range: {result.get('date_range', 'N/A')}")
    print(f"  Total days: {result.get('total_days', 'N/A'):,}")
    
    if output_file_path.exists():
        df_check = pd.read_csv(output_file_path)
        print(f"  Output file: {output_file_path.name}")
        print(f"  Output file size: {output_file_path.stat().st_size / 1024:.2f} KB")
        print(f"  Output columns: {', '.join(df_check.columns)}")
        print(f"  Output data:")
        print(df_check.to_string(index=False))


SAMPLE VERIFICATION

Station: 3
  Input file: produkt_tu_stunde_19500401_20110331_00003_lat_50_7827_lon_6_0941.csv
  Design Temperature: -8.58°C
  Date range: 20030101 to 20110331
  Total days: 3,012
  Output file: produkt_tu_stunde_19500401_20110331_00003_lat_50_7827_lon_6_0941.csv
  Output file size: 0.26 KB
  Output columns: STATIONS_ID, latitude, longitude, design_temperature, lowest_10_rolling_avgs, date_range_start, date_range_end, total_days, days_with_rolling_avg
  Output data:
 STATIONS_ID  latitude  longitude  design_temperature                                               lowest_10_rolling_avgs  date_range_start  date_range_end  total_days  days_with_rolling_avg
           3   50.7827     6.0941            -8.58125 -8.58, -8.26, -7.92, -7.41, -7.36, -7.03, -6.32, -6.29, -6.21, -5.86          20030101        20110331        3012                   3011

Station: 3_2
  Input file: produkt_tu_stunde_19500401_20110331_3_2_lat_50_7827_lon_6_0941.csv
  Design Temperature: -8.58°C